# Cocoa Contamination Hackathon — YOLO11 (T4)

Self-contained training + inference notebook for the Zindi IndabaX SA Cocoa Contamination Hackathon.

**Metric:** mAP@IoU0.5 &nbsp;|&nbsp; **Limits:** single T4, &le;9h train, &le;3h infer &nbsp;|&nbsp; **Edge:** ONNX export.

### Before running
1. Settings &rarr; **Accelerator = GPU T4 x1**, **Internet = On**.
2. Add the data. Either:
   - Upload `dataset.zip` (the ~9.4 GB images) as a Kaggle Dataset and attach it, **or**
   - Let cell 2 download it from the public URL.
3. Run all cells. Final output: `submission.csv` + `best.onnx`.

In [1]:
# 1. Install deps
!pip -q install "ultralytics>=8.3.0" onnx onnxslim onnxruntime
import ultralytics, torch
print('ultralytics', ultralytics.__version__, '| CUDA', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

ultralytics 8.4.90 | CUDA False 


In [ ]:
# 2. Locate or download the dataset
#    The archive already contains YOLO labels: images/{train,test}/ and labels/train/
import os, glob, zipfile, urllib.request
from pathlib import Path

DATA = Path('/kaggle/working/data')
DATA.mkdir(parents=True, exist_ok=True)

# 1) Prefer an already-extracted attached dataset (images/ + labels/)
attached = glob.glob('/kaggle/input/**/labels/train', recursive=True)
# 2) Else look for an attached OR previously-downloaded dataset.zip
found_zip = (glob.glob('/kaggle/input/**/dataset.zip', recursive=True)
             + ([str(DATA / 'dataset.zip')] if (DATA / 'dataset.zip').exists() else []))

if attached:
    SRC = Path(attached[0]).parents[1]   # folder containing images/ and labels/
    print('Using extracted dataset at', SRC)
else:
    zip_path = Path(found_zip[0]) if found_zip else (DATA / 'dataset.zip')
    if not zip_path.exists():
        print('No local/attached dataset.zip found — downloading (~9.4 GB)...')
        urllib.request.urlretrieve('https://storage.googleapis.com/cocoa/dataset.zip', zip_path)
    else:
        print('Using existing dataset.zip at', zip_path, '(no download)')
    print('Extracting...')
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(DATA)
    SRC = DATA

print('train imgs:', len(glob.glob(str(SRC/'images/train/*'))),
      '| test imgs:', len(glob.glob(str(SRC/'images/test/*'))),
      '| labels:', len(glob.glob(str(SRC/'labels/train/*.txt'))))

KeyboardInterrupt: 

In [ ]:
# 3. Build a seeded 85/15 train/val split (uses provided YOLO labels)
import os, random, shutil
from pathlib import Path

CLASSES = ['anthracnose', 'cssvd', 'healthy']   # index order = provided labels (0,1,2)
OUT = Path('/kaggle/working/yolo')
SRC_IMG, SRC_LBL = SRC/'images/train', SRC/'labels/train'
VAL_FRACTION, SEED = 0.15, 42

def find_image(stem):
    for ext in ('.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG'):
        p = SRC_IMG / f'{stem}{ext}'
        if p.exists():
            return p
    return None

def link_or_copy(src, dst):
    if dst.exists():
        return
    try:
        os.link(src, dst)
    except OSError:
        shutil.copy2(src, dst)

stems = sorted(p.stem for p in SRC_LBL.glob('*.txt'))
random.Random(SEED).shuffle(stems)
n_val = int(len(stems) * VAL_FRACTION)
splits = {'val': stems[:n_val], 'train': stems[n_val:]}

for sub in ('images', 'labels'):
    for split in ('train', 'val'):
        (OUT/sub/split).mkdir(parents=True, exist_ok=True)

counts = {'train': 0, 'val': 0}
for split, ids in splits.items():
    for stem in ids:
        img = find_image(stem)
        lbl = SRC_LBL / f'{stem}.txt'
        if img is None or not lbl.exists():
            continue
        link_or_copy(img, OUT/'images'/split/img.name)
        link_or_copy(lbl, OUT/'labels'/split/lbl.name)
        counts[split] += 1

(OUT/'cocoa.yaml').write_text(
    f'path: {OUT.as_posix()}\ntrain: images/train\nval: images/val\n'
    f'nc: {len(CLASSES)}\nnames: {CLASSES}\n')
print('train', counts['train'], '| val', counts['val'], '| classes', CLASSES)

In [ ]:
# 4. Train (yolo11s @ 640). Fits the T4 / 9h budget. Auto-exports ONNX.
from ultralytics import YOLO

MODEL, IMGSZ, EPOCHS, NAME = 'yolo11s.pt', 640, 100, 'cocoa_yolo11s'

model = YOLO(MODEL)
results = model.train(
    data=str(OUT/'cocoa.yaml'), imgsz=IMGSZ, epochs=EPOCHS, batch=-1,
    patience=25, project='/kaggle/working/runs', name=NAME, seed=42,
    cache='ram', cos_lr=True, close_mosaic=10,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, fliplr=0.5, flipud=0.2,
    mosaic=1.0, mixup=0.1, scale=0.5, amp=True, plots=True,
)
best = Path(results.save_dir) / 'weights' / 'best.pt'
print('best weights:', best)
YOLO(str(best)).export(format='onnx', imgsz=IMGSZ, opset=12, simplify=True)
print('ONNX exported next to best.pt')

In [ ]:
# 5. Inference -> submission.csv (format: Image_ID,class,confidence,ymin,xmin,ymax,xmax)
import csv
from ultralytics import YOLO

CONF, MAX_DET, TTA = 0.05, 100, True
TEST_CSV = glob.glob(str(SRC/'..'/'Test.csv')) + glob.glob('/kaggle/input/**/Test.csv', recursive=True) + glob.glob(str(DATA/'Test.csv'))
TEST_CSV = next((p for p in TEST_CSV if Path(p).exists()), None)
assert TEST_CSV, 'Test.csv not found — attach it or place in data/'
IMG_DIR = SRC/'images/test'

with open(TEST_CSV, newline='') as f:
    test_ids = [r['Image_ID'] for r in csv.DictReader(f)]

model = YOLO(str(best))
rows, seen = [], set()
for img_id in test_ids:
    path = IMG_DIR / img_id
    if not path.exists():
        continue
    r = model.predict(source=str(path), imgsz=IMGSZ, conf=CONF, max_det=MAX_DET,
                      augment=TTA, verbose=False)[0]
    for box in r.boxes:
        cls = CLASSES[int(box.cls)]
        conf = float(box.conf)
        x1, y1, x2, y2 = (float(v) for v in box.xyxy[0])
        rows.append([img_id, cls, round(conf, 6), round(y1), round(x1), round(y2), round(x2)])
        seen.add(img_id)

for img_id in test_ids:            # placeholder for zero-detection images
    if img_id not in seen:
        rows.append([img_id, 'healthy', 0.01, 0, 0, 1, 1])

with open('/kaggle/working/submission.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['Image_ID', 'class', 'confidence', 'ymin', 'xmin', 'ymax', 'xmax'])
    w.writerows(rows)
print('wrote', len(rows), 'rows for', len(test_ids), 'images -> /kaggle/working/submission.csv')